In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Master Initialization
raw_schema = "order_id string, store_id string, user_id int, raw_amount string, order_date string"
raw_data = [
    ("ORD1", "S1", 100, " 1500.50 ", "2026-03-10"),
    ("ORD2", "S2", 101, "invalid", "2026-03-11"),
    ("ORD3", "S1", 102, "  900.00", "2026-03-12"),
    ("ORD4", "S3", 100, "300.25", "2026-03-13"),
    ("ORD5", "S1", 100, None, "2026-03-13")
]
raw_orders = spark.createDataFrame(raw_data, schema=raw_schema)

catalog_schema = "store_id string, location string, is_flagship int"
catalog_data = [
    ("S1", "New York", 1),
    ("S2", "London", 0)
]
store_catalog = spark.createDataFrame(catalog_data, schema=catalog_schema)

promo_schema = "promo_code string, discount_rate double, start_date string, end_date string"
promo_data = [
    ("MARCH_MADNESS", 0.10, "2026-03-01", "2026-03-15")
]
promotions = spark.createDataFrame(promo_data, schema=promo_schema)

In [0]:
raw_orders.display()

order_id,store_id,user_id,raw_amount,order_date
ORD1,S1,100,1500.50,2026-03-10
ORD2,S2,101,invalid,2026-03-11
ORD3,S1,102,900.00,2026-03-12
ORD4,S3,100,300.25,2026-03-13
ORD5,S1,100,null,2026-03-13


### Step 1: Clean and Cast Raw Orders
1. Trim whitespace from `raw_amount`.
2. Try cast `raw_amount` to a `double` named `clean_amount`.
3. Drop rows where `clean_amount` is null after casting.

* **Expected Result:**
|order_id|store_id|user_id|clean_amount|order_date|
|---|---|---|---|---|
|ORD1|S1|100|1500.5|2026-03-10|
|ORD3|S1|102|900|2026-03-12|
|ORD4|S3|100|300.25|2026-03-13|

In [0]:
df_1 = (
    raw_orders
    .withColumn("clean_amount",trim(col("raw_amount")).try_cast("double"))
    .dropna(subset=["clean_amount"])
    .select("order_id","store_id","user_id","clean_amount","order_date")
)

df_1.display()


order_id,store_id,user_id,clean_amount,order_date
ORD1,S1,100,1500.5,2026-03-10
ORD3,S1,102,900.0,2026-03-12
ORD4,S3,100,300.25,2026-03-13


### Step 2: Left Join Store Catalog & Coalesce
1. Perform a left join of `step1_df` with `store_catalog` on `store_id`.
2. Use `coalesce` to replace null `location` with "Online".
3. Use `coalesce` to replace null `is_flagship` with `0` and cast to boolean.

* **Expected Result:**
|store_id|order_id|user_id|clean_amount|order_date|location|is_flagship|
|---|---|---|---|---|---|---|
|S1|ORD1|100|1500.5|2026-03-10|New York|true|
|S1|ORD3|102|900|2026-03-12|New York|true|
|S3|ORD4|100|300.25|2026-03-13|Online|false|

In [0]:
df_2=(
    df_1.alias("df1")
    .join(store_catalog.alias("store"),["store_id"],"left")
    .withColumn("location", coalesce(col("store.location"), lit("Online")))
    .withColumn("is_flagship",coalesce(col("store.is_flagship"),lit(0)).cast("boolean"))
    )
df_2.display()

store_id,order_id,user_id,clean_amount,order_date,location,is_flagship
S1,ORD1,100,1500.5,2026-03-10,New York,true
S1,ORD3,102,900.0,2026-03-12,New York,true
S3,ORD4,100,300.25,2026-03-13,Online,false


### Step 3: Date Between Join for Promotions
1. Join `step2_df` and `promotions` based on an open condition (cross logic but filtered). 
2. Filter the join where `order_date` is between `start_date` and `end_date`.
3. Calculate `final_amount` = `clean_amount * (1 - discount_rate)`.

* **Expected Result:**
|store_id|order_id|user_id|clean_amount|order_date|location|is_flagship|promo_code|final_amount|
|---|---|---|---|---|---|---|---|---|
|S1|ORD1|100|1500.5|2026-03-10|New York|true|MARCH_MADNESS|1350.45|
|S1|ORD3|102|900|2026-03-12|New York|true|MARCH_MADNESS|810|
|S3|ORD4|100|300.25|2026-03-13|Online|false|MARCH_MADNESS|270.225|

In [0]:
df_3 = (
    df_2.alias("df2")
    .join(promotions.alias("promo")
          ,col("df2.order_date").between(col("promo.start_date"),col("promo.end_date"))
          ,how="left")
    .withColumn("final_amount",col("clean_amount") * (1 - col("discount_rate")))
    )
df_3.display()


store_id,order_id,user_id,clean_amount,order_date,location,is_flagship,promo_code,discount_rate,start_date,end_date,final_amount
S1,ORD1,100,1500.5,2026-03-10,New York,true,MARCH_MADNESS,0.1,2026-03-01,2026-03-15,1350.45
S1,ORD3,102,900.0,2026-03-12,New York,true,MARCH_MADNESS,0.1,2026-03-01,2026-03-15,810.0
S3,ORD4,100,300.25,2026-03-13,Online,false,MARCH_MADNESS,0.1,2026-03-01,2026-03-15,270.225


### Step 4: Window Functions
1. Create a Window partitioned by `user_id` and ordered by `order_date` descending.
2. Calculate the `row_number()` to find the latest order per user (row_num = 1).
3. Calculate the `sum` of `final_amount` per user as `lifetime_value`.

* **Expected Result:**
|store_id|order_id|user_id|clean_amount|order_date|location|is_flagship|promo_code|final_amount|latest_order_rank|lifetime_value|
|---|---|---|---|---|---|---|---|---|---|---|
|S3|ORD4|100|300.25|2026-03-13|Online|false|MARCH_MADNESS|270.225|1|1620.6750000000002|
|S1|ORD1|100|1500.5|2026-03-10|New York|true|MARCH_MADNESS|1350.45|2|1620.6750000000002|
|S1|ORD3|102|900|2026-03-12|New York|true|MARCH_MADNESS|810|1|810|

In [0]:
row_window = Window.partitionBy("user_id").orderBy(col("order_date").desc())
sum_window = Window.partitionBy("user_id")

df_4 = (
    df_3
    .withColumn("latest_order_rank",row_number().over(row_window))
    .withColumn("lifetime_value",sum("final_amount").over(sum_window))
    )
df_4.display()


store_id,order_id,user_id,clean_amount,order_date,location,is_flagship,promo_code,discount_rate,start_date,end_date,final_amount,latest_order_rank,lifetime_value
S3,ORD4,100,300.25,2026-03-13,Online,false,MARCH_MADNESS,0.1,2026-03-01,2026-03-15,270.225,1,1620.6750000000002
S1,ORD1,100,1500.5,2026-03-10,New York,true,MARCH_MADNESS,0.1,2026-03-01,2026-03-15,1350.45,2,1620.6750000000002
S1,ORD3,102,900.0,2026-03-12,New York,true,MARCH_MADNESS,0.1,2026-03-01,2026-03-15,810.0,1,810.0


### Step 5: Struct Generation and Writing Output
1. Filter `step4_df` for only `latest_order_rank == 1`.
2. Create a struct named `user_profile` containing `location`, `lifetime_value`, and `is_flagship`.
3. Write the final DataFrame to table name `session_7_spark.homework_final` with option `overwriteSchema`.

* **Expected Result:**
|user_id|order_id|user_profile|
|---|---|---|
|100|ORD4|{"location":"Online","lifetime_value":1620.6750000000002,"is_flagship":false}|
|102|ORD3|{"location":"New York","lifetime_value":810,"is_flagship":true}|

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS session_7_spark;

In [0]:
df_5 = (
    df_4
    .filter(col("latest_order_rank") == 1)
    .select(
        col("user_id")
        ,col("order_id")
        ,struct(
            col("location"),
            col("lifetime_value"),
            col("is_flagship")
        ))
)

(df_5.write
  .mode("overwrite")
  .option("overwriteSchema","true")
  .saveAsTable("session_7_spark.homework_final")
)
df_5.display()


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5159250740739360>, line 17
      1 df_5 = (
      2     df_4
      3     .filter(col("latest_order_rank") == 1)
   (...)
     11         ))
     12 )
     14 (df_5.write
     15   .mode("overwrite")
     16   .option("overwriteSchema","true")
---> 17   .saveAsTable("session_7_spark.homework_final")
     18 )
     19 df_5.display()

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/readwriter.py:713, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    711 self._write.table_name = name
    712 self._write.table_save_method = "save_as_table"
--> 713 _, _, ei = self._spark.client.execute_command(
    714     self._write.command(self._spark.client), self._write.observations
    715 )
    716 self._callback(ei)

File /databricks/python/lib/python3.11/site-packages/pyspark/s